In [12]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ============================================================
# 2. ZIP FILE PATH
# ============================================================

# Change this to your ZIP file location
zip_path = "/archive.zip"
# Folder where the ZIP will be extracted
extract_path = "/content/dataset"


# ============================================================
# 3. EXTRACT ZIP FILE
# ============================================================

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP file extracted successfully!")


# ============================================================
# 4. FIND TRAIN AND TEST FOLDERS
# ============================================================

train_dir = None
test_dir = None

for root, dirs, files in os.walk(extract_path):

    for d in dirs:
        if d.lower() == "train":
            train_dir = os.path.join(root, d)

        if d.lower() == "test":
            test_dir = os.path.join(root, d)

print("Train folder:", train_dir)
print("Test folder :", test_dir)


# ============================================================
# 5. CHECK FOLDER STRUCTURE
# ============================================================

print("\nTrain classes:")

train_classes = [
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
]

for c in train_classes:
    print(c)

print("\nTest classes:")

test_classes = [
    d for d in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, d))
]

for c in test_classes:
    print(c)

print("\nNumber of classes:", len(train_classes))


ZIP file extracted successfully!
Train folder: /content/dataset/Combined Dataset/train
Test folder : /content/dataset/Combined Dataset/test

Train classes:
Moderate Impairment
Very Mild Impairment
No Impairment
Mild Impairment

Test classes:
Moderate Impairment
Very Mild Impairment
No Impairment
Mild Impairment

Number of classes: 4


In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.4,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    rotation_range=90,
    horizontal_flip=True,
    fill_mode='nearest'
)



In [14]:
# Test data generator
test_datagen = ImageDataGenerator(
    rescale=1./255
)

# ============================================================
# LOAD TRAIN IMAGES
# ============================================================

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=32,
    shuffle=True
)


# ============================================================
# LOAD TEST IMAGES
# ============================================================

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)


# ============================================================
# CHECK CLASSES
# ============================================================

print("Class indices:")
print(train_generator.class_indices)

print("Number of classes:",
      len(train_generator.class_indices))

Found 10240 images belonging to 4 classes.
Found 1279 images belonging to 4 classes.
Class indices:
{'Mild Impairment': 0, 'Moderate Impairment': 1, 'No Impairment': 2, 'Very Mild Impairment': 3}
Number of classes: 4


In [15]:
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import Flatten
from keras.layers import MaxPooling2D
from keras.layers import Dense
import numpy as np

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# VGG-16 style model implemented as a Sequential model
# Matches the original VGG-16 block structure shown in your snippet

input_shape = (128, 128, 1)  # change if you need different input size
num_classes = 4  # change to the number of output classes you need

model = Sequential()
# Block 1
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same',
                 activation='relu', input_shape=input_shape))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

# Block 2
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
# Block 3
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

# Block 4
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
# Block 5
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding='same',
                 activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

# Classification head
model.add(Flatten())
model.add(Dense(units=4096, activation='relu'))
model.add(Dense(units=4096, activation='relu'))
model.add(Dense(units=num_classes, activation='softmax'))

# Compile (use appropriate loss for your labels)
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Display summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_26 (Conv2D)              │ (None, 128, 128, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_28 (Conv2D)              │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_34 (Conv2D)              │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_35 (Conv2D)              │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_36 (Conv2D)              │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_37 (Conv2D)              │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_38 (Conv2D)              │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 4096)           │    33,558,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │        16,388 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,069,764 (248.22 MB)

 Trainable params: 65,069,764 (248.22 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.fit(train_generator, epochs=1)

320/320 ━━━━━━━━━━━━━━━━━━━━ 91s 186ms/step - accuracy: 0.2421 - loss: 1.3930


In [18]:
# Evaluate the model on the test data
loss, accuracy_vgg = model.evaluate(test_generator)

# Print the accuracy
print(f"Test Accuracy of VGG16 Original: {accuracy_vgg}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 240ms/step - accuracy: 0.5004 - loss: 1.3830
Test Accuracy of VGG16 Original: 0.5003909468650818


In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential()
# Block 1
model.add(Conv2D(64, (3,3), padding="same", activation="relu", input_shape=(128,128,1)))
model.add(BatchNormalization())
model.add(Conv2D(64, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))

# Block 2
model.add(Conv2D(128, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(128, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
# Block 3
model.add(Conv2D(256, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(256, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(256, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))

# Block 4 (you *can* drop one of the 512 blocks if data is small)
model.add(Conv2D(512, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(512, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(512, (3,3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
# Global part
model.add(Flatten())
model.add(Dense(1024, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(512, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(4, activation="softmax"))

In [24]:
from tensorflow.keras.layers import GlobalAveragePooling2D

# ... conv blocks as before ...

model.add(GlobalAveragePooling2D())
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(4, activation="softmax"))

ValueError: Input 0 with name 'None' of layer 'global_average_pooling2d' is incompatible with the layer: expected ndim=4, found ndim=2. Full shape received: (None, 4)

In [22]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Concatenate, Flatten, Dense, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# 1. Input for grayscale image
input_tensor = Input(shape=(128, 128, 1))

# 2. Convert grayscale -> RGB by concatenation
input_tensor_rgb = Concatenate()([input_tensor, input_tensor, input_tensor])

# Now shape becomes (128, 128, 3)

In [23]:
# 1. Input for grayscale image
input_tensor = Input(shape=(128, 128, 1))

# 2. Convert grayscale → RGB by concatenation
input_tensor_rgb = Concatenate()([input_tensor, input_tensor, input_tensor])

# Now shape becomes (128, 128, 3)

# 3. Load VGG16 WITHOUT top layers
vgg16 = VGG16(
    weights='imagenet',
    include_top=False,
    input_tensor=input_tensor_rgb
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [25]:
# 4. Create new model and add VGG as base
modelvgg16 = Sequential()
modelvgg16.add(vgg16)

# 5. Add custom classification head
modelvgg16.add(Flatten())
modelvgg16.add(Dense(256, activation='relu'))
modelvgg16.add(Dropout(0.5))
modelvgg16.add(Dense(128, activation='relu'))
modelvgg16.add(Dropout(0.5))

In [26]:
# Final output for 4 classes
modelvgg16.add(Dense(4, activation='softmax'))

# 6. Freeze VGG16 layers (transfer learning)
for layer in vgg16.layers:
    layer.trainable = False

# 7. Compile model
modelvgg16.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

modelvgg16.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,845,508 (64.26 MB)

 Trainable params: 2,130,820 (8.13 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [28]:
modelvgg16.fit(train_generator, epochs=5)

Epoch 1/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 30s 93ms/step - accuracy: 0.3661 - loss: 1.3005
Epoch 2/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 29s 90ms/step - accuracy: 0.4327 - loss: 1.2205
Epoch 3/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 28s 88ms/step - accuracy: 0.4763 - loss: 1.1527
Epoch 4/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 28s 88ms/step - accuracy: 0.4962 - loss: 1.1119
Epoch 5/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 29s 91ms/step - accuracy: 0.5134 - loss: 1.0717


In [33]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, MaxPooling2D,
    Input, Concatenate, Flatten, BatchNormalization
)
from tensorflow.keras.optimizers import Adam

# 1-channel → 3-channel
input_tensor = Input(shape=(128,128,1))

input_rgb = Concatenate()(
    [input_tensor, input_tensor, input_tensor]
)

# Load VGG16
vgg16 = VGG16(
    weights='imagenet',
    include_top=False,
    input_tensor=input_rgb
)

# Freeze VGG16
vgg16.trainable = False

# Create model
modelvgg16_plus_2 = Sequential()

modelvgg16_plus_2.add(vgg16)

# Custom Conv Block 1
modelvgg16_plus_2.add(
    Conv2D(128, (3,3), padding='same', activation='relu')
)
modelvgg16_plus_2.add(BatchNormalization())
modelvgg16_plus_2.add(MaxPooling2D((2,2)))

# Custom Conv Block 2
modelvgg16_plus_2.add(
    Conv2D(128, (3,3), padding='same', activation='relu')
)
modelvgg16_plus_2.add(BatchNormalization())

# Flatten
modelvgg16_plus_2.add(Flatten())

# Dense layers
modelvgg16_plus_2.add(Dense(256, activation='relu'))
modelvgg16_plus_2.add(Dropout(0.5))

modelvgg16_plus_2.add(Dense(128, activation='relu'))
modelvgg16_plus_2.add(Dropout(0.5))

# Output
modelvgg16_plus_2.add(Dense(4, activation='softmax'))

# Compile
modelvgg16_plus_2.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

modelvgg16_plus_2.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_53 (Conv2D)              │ (None, 4, 4, 128)      │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 4, 4, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_54 (Conv2D)              │ (None, 2, 2, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 2, 2, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,617,988 (59.58 MB)

 Trainable params: 902,788 (3.44 MB)

 Non-trainable params: 14,715,200 (56.13 MB)

In [34]:
modelvgg16_plus_2.fit(train_generator, epochs=5)

Epoch 1/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 36s 90ms/step - accuracy: 0.3026 - loss: 1.4719
Epoch 2/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 29s 89ms/step - accuracy: 0.4082 - loss: 1.2607
Epoch 3/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 28s 89ms/step - accuracy: 0.4920 - loss: 1.1220
Epoch 4/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 28s 89ms/step - accuracy: 0.5372 - loss: 1.0371
Epoch 5/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 28s 89ms/step - accuracy: 0.5600 - loss: 0.9789


In [40]:
n_classes=4


In [42]:
# ---------------- CNN1 ----------------
CNN1 = Sequential()

CNN1.add(Conv2D(256, (3, 3), activation='relu', input_shape=input_shape))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))

CNN1.add(Conv2D(64, (3, 3), activation='relu'))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))

CNN1.add(Conv2D(256, (3, 3), activation='relu'))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))

CNN1.add(Conv2D(128, (3, 3), activation='relu'))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))
CNN1.add(Conv2D(256, (3, 3), activation='relu'))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))

CNN1.add(Flatten())
CNN1.add(Dense(128, activation='relu'))
CNN1.add(Dropout(0.2))
CNN1.add(Dense(64, activation='relu'))
CNN1.add(Dense(n_classes, activation='softmax'))

CNN1.compile(
    optimizer='adam',
loss='categorical_crossentropy',
    metrics=['accuracy']
)

CNN1.summary()

CNN1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
CNN1.fit(
    train_generator,
    epochs=5,
    validation_data=test_generator
)

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_65 (Conv2D)              │ (None, 126, 126, 256)  │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_33 (MaxPooling2D) │ (None, 63, 63, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_66 (Conv2D)              │ (None, 61, 61, 64)     │       147,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_34 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_67 (Conv2D)              │ (None, 28, 28, 256)    │       147,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_35 (MaxPooling2D) │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_68 (Conv2D)              │ (None, 12, 12, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_36 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_69 (Conv2D)              │ (None, 4, 4, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_37 (MaxPooling2D) │ (None, 2, 2, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027,716 (3.92 MB)

 Trainable params: 1,027,716 (3.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 38s 100ms/step - accuracy: 0.2468 - loss: 1.3869 - val_accuracy: 0.1400 - val_loss: 1.3916
Epoch 2/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 32s 99ms/step - accuracy: 0.2465 - loss: 1.3865 - val_accuracy: 0.3503 - val_loss: 1.3861
Epoch 3/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 32s 98ms/step - accuracy: 0.2386 - loss: 1.3865 - val_accuracy: 0.3503 - val_loss: 1.3860
Epoch 4/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 31s 97ms/step - accuracy: 0.2464 - loss: 1.3864 - val_accuracy: 0.1400 - val_loss: 1.3861
Epoch 5/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 32s 100ms/step - accuracy: 0.2424 - loss: 1.3864 - val_accuracy: 0.1400 - val_loss: 1.3879


In [43]:
# ======================== EVALUATE ========================

loss, acc = CNN1.evaluate(test_generator)

print(f"\nCNN1 Test Accuracy: {acc:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.1400 - loss: 1.3879

CNN1 Test Accuracy: 0.1400


In [48]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout
)

# Define input and number of classes
input_shape = (128, 128, 1)
n_classes = 4

# Create CNN2
CNN2 = Sequential([

    Conv2D(64, (3,3), activation='relu',
           padding='same', input_shape=input_shape),
    BatchNormalization(),

    Conv2D(64, (3,3), activation='relu',
           padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu',
           padding='same'),
    BatchNormalization(),

    Conv2D(128, (3,3), activation='relu',
           padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(256, (3,3), activation='relu',
           padding='same'),
    MaxPooling2D((2,2)),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.4),

    Dense(128, activation='relu'),

    Dense(n_classes, activation='softmax')
])

# Compile
CNN2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Show model
CNN2.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_70 (Conv2D)              │ (None, 128, 128, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_71 (Conv2D)              │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_38 (MaxPooling2D) │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_72 (Conv2D)              │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_73 (Conv2D)              │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_74 (Conv2D)              │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_10 (Flatten)            │ (None, 65536)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 256)            │    16,777,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,365,828 (66.25 MB)

 Trainable params: 17,365,444 (66.24 MB)

 Non-trainable params: 384 (1.50 KB)

In [49]:
# ======================== EVALUATE ========================

loss, acc = CNN2.evaluate(test_generator)

print(f"\nCNN2 Test Accuracy: {acc:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.5004 - loss: 1.3796

CNN2 Test Accuracy: 0.5004


In [51]:
# ---------------- CNN3 ----------------
CNN3 = Sequential([
    Conv2D(32, (5,5), activation='relu', padding='same', input_shape=input_shape),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu', padding='same'),
MaxPooling2D((2,2)),

Flatten(),
Dense(128, activation='relu'),
Dropout(0.5),
Dense(n_classes, activation='softmax')
])

CNN3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
CNN3.summary()

CNN3.fit(
    train_generator,
    epochs=5,
    validation_data=test_generator
)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_79 (Conv2D)              │ (None, 128, 128, 32)   │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_45 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_80 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_46 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_81 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_47 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_82 (Conv2D)              │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_48 (MaxPooling2D) │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,290,500 (4.92 MB)

 Trainable params: 1,290,244 (4.92 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 34s 89ms/step - accuracy: 0.3542 - loss: 1.3075 - val_accuracy: 0.5004 - val_loss: 1.3340
Epoch 2/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - accuracy: 0.5398 - loss: 1.0479 - val_accuracy: 0.2306 - val_loss: 1.6300
Epoch 3/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 27s 83ms/step - accuracy: 0.5959 - loss: 0.9322 - val_accuracy: 0.2072 - val_loss: 3.2104
Epoch 4/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - accuracy: 0.6450 - loss: 0.7884 - val_accuracy: 0.5293 - val_loss: 0.9224
Epoch 5/5
320/320 ━━━━━━━━━━━━━━━━━━━━ 27s 83ms/step - accuracy: 0.6644 - loss: 0.7277 - val_accuracy: 0.5301 - val_loss: 0.9295


In [52]:
# ======================== EVALUATE ========================

loss, acc = CNN3.evaluate(test_generator)

print(f"\nCNN3 Test Accuracy: {acc:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5301 - loss: 0.9295

CNN3 Test Accuracy: 0.5301


In [53]:
# This is the correct stacking behavior.

for base_model in [CNN1, CNN2, CNN3]:
    base_model.trainable = False

# Shared input
stack_input = Input(shape=input_shape)

# Get softmax outputs from each base model
pred1 = CNN1(stack_input)
pred2 = CNN2(stack_input)
pred3 = CNN3(stack_input)

# Concatenate predictions → meta-features
merged = Concatenate()([pred1, pred2, pred3])  # shape: (None, 3 * n_classes)
meta_input_dim = 3 * n_classes  # e.g., 3 * 4 = 12

MLP_meta = Sequential(name="MLP_meta")
MLP_meta.add(Dense(32, activation='relu', input_shape=(meta_input_dim,)))
MLP_meta.add(Dense(32, activation='relu'))
MLP_meta.add(Dropout(0.5))
MLP_meta.add(Dense(n_classes, activation='softmax'))

# Pass merged predictions to MLP_meta
stack_output = MLP_meta(merged)
# Final stacking model
Stacking_Model = Model(
    inputs=stack_input,
    outputs=stack_output
)

Stacking_Model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Stacking_Model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "functional_181"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_17      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_9        │ (None, 4)         │  1,027,716 │ input_layer_17[0… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_10       │ (None, 4)         │ 17,365,828 │ input_layer_17[0… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_12       │ (None, 4)         │  1,290,500 │ input_layer_17[0… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 12)        │          0 │ sequential_9[0][… │
│ (Concatenate)       │                   │            │ sequential_10[0]… │
│                     │                   │            │ sequential_12[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MLP_meta            │ (None, 4)         │      1,604 │ concatenate_4[0]… │
│ (Sequential)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,685,648 (75.09 MB)

 Trainable params: 1,604 (6.27 KB)

 Non-trainable params: 19,684,044 (75.09 MB)

In [54]:
# ======================== TRAIN STACKING MODEL ========================

# Now train the stacking ensemble using the generators
epochs_stack = 3

history_stack = Stacking_Model.fit(
    train_generator,
    epochs=epochs_stack,
    validation_data=test_generator
)

# ======================== EVALUATE ========================

loss, acc = Stacking_Model.evaluate(test_generator)

print(f"\nStacking Model Test Accuracy: {acc:.4f}")

Epoch 1/3
320/320 ━━━━━━━━━━━━━━━━━━━━ 38s 99ms/step - accuracy: 0.5261 - loss: 1.1059 - val_accuracy: 0.5575 - val_loss: 1.0403
Epoch 2/3
320/320 ━━━━━━━━━━━━━━━━━━━━ 28s 89ms/step - accuracy: 0.5986 - loss: 0.9180 - val_accuracy: 0.5551 - val_loss: 1.0257
Epoch 3/3
320/320 ━━━━━━━━━━━━━━━━━━━━ 29s 89ms/step - accuracy: 0.6109 - loss: 0.8870 - val_accuracy: 0.5598 - val_loss: 1.0078
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.5598 - loss: 1.0078

Stacking Model Test Accuracy: 0.5598


In [55]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout# Define the CNN model
CNN1 = Sequential()

# --------------------------------------------------
# Convolutional Layer 1
# --------------------------------------------------
CNN1.add(Conv2D(
    256,
    (3, 3),
    activation='relu',
    padding='same',
    input_shape=(128, 128, 1)
))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))


# --------------------------------------------------
# Convolutional Layer 2
# --------------------------------------------------
CNN1.add(Conv2D(
    64,
    (3, 3),
    activation='relu'
))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))


# --------------------------------------------------
# Convolutional Layer 3
# --------------------------------------------------
CNN1.add(Conv2D(
    256,
    (3, 3),
    activation='relu'
))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))


# --------------------------------------------------
# Convolutional Layer 4
# --------------------------------------------------
CNN1.add(Conv2D(
    128,
    (3, 3),
    activation='relu'
))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))


# --------------------------------------------------
# Convolutional Layer 5
# --------------------------------------------------
CNN1.add(Conv2D(
    256,
    (3, 3),
    activation='relu'
))
CNN1.add(MaxPooling2D(pool_size=(2, 2)))


# --------------------------------------------------
# Flatten
# --------------------------------------------------
CNN1.add(Flatten())


# --------------------------------------------------
# Fully Connected Layer
# --------------------------------------------------
CNN1.add(Dense(128, activation='relu'))

# Dropout
CNN1.add(Dropout(0.5))


# --------------------------------------------------
# Output Layer
# --------------------------------------------------
CNN1.add(Dense(
    len(train_generator.class_indices),
    activation='softmax'
))

In [56]:
CNN1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
CNN1.fit(train_generator, epochs=2)

Epoch 1/2
320/320 ━━━━━━━━━━━━━━━━━━━━ 42s 98ms/step - accuracy: 0.2497 - loss: 1.3869
Epoch 2/2
320/320 ━━━━━━━━━━━━━━━━━━━━ 30s 94ms/step - accuracy: 0.2456 - loss: 1.3865
